# Depth Anything V2 benchmark-sensitivity reproduction

Run on a free Colab T4. This notebook is a thin runner around `src/` -- all the actual logic (metrics, alignment, decision rule) lives in the repository and is unit-tested offline; see `tests/test_metrics.py`.

Steps: clone repos -> install deps -> download checkpoints/data (after the DA-2K licence check) -> `pytest` sanity check -> sign-convention calibration -> 20-image smoke test -> full run -> decision rule.

In [ ]:
!git clone https://github.com/SiboneloMlambo/Computer_Vision_Assignment_MSc.git repo
%cd repo
!git clone https://github.com/LiheYoung/Depth-Anything.git external/Depth-Anything
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git external/Depth-Anything-V2
!pip install -q -r external/Depth-Anything/requirements.txt
!pip install -q -r external/Depth-Anything-V2/requirements.txt
!pip install -q -r requirements.txt

## Sanity check the student-written code first (no GPU, no checkpoints needed)

In [ ]:
!pytest tests/ -v

## Download checkpoints

Fill in the official release URLs for V1-S and V2-S. Record them in `run_log.md`, not just here.

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)
# V2-S, official Depth Anything V2 release
!wget -O checkpoints/depth_anything_v2_vits.pth "https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth"
# V1-S, official Depth Anything V1 release
!wget -O checkpoints/depth_anything_v1_vits.pth "https://huggingface.co/spaces/LiheYoung/Depth-Anything/resolve/main/checkpoints/depth_anything_vits14.pth"

## Data: read `data/README.md` and complete the licence/provenance check BEFORE running the next cell

In [ ]:
# Download DA-2K and DIODE validation data.
import os
import shutil
import tarfile
import zipfile
from pathlib import Path

!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

# --- DA-2K ---
# Extract into a temporary directory first. The repository already contains
# data/da2k/README.md, so moving a folder directly to data/da2k would create
# an incorrect nested directory.
data_root = Path("data")
target = data_root / "da2k"
staging = data_root / "da2k_extracted"
target.mkdir(parents=True, exist_ok=True)

if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)

da2k_zip = hf_hub_download(
    repo_id="depth-anything/DA-2K",
    filename="DA-2K.zip",
    repo_type="dataset",
)
with zipfile.ZipFile(da2k_zip) as archive:
    archive.extractall(staging)

annotation_files = list(staging.rglob("annotations.json"))
assert len(annotation_files) == 1, (
    f"Expected one annotations.json, found {len(annotation_files)}: "
    f"{annotation_files}"
)
dataset_root = annotation_files[0].parent

# Merge the actual dataset root into the existing data/da2k directory.
for source in dataset_root.iterdir():
    destination = target / source.name
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(source, destination)

shutil.rmtree(staging)
assert (target / "annotations.json").is_file()
print("DA-2K ready:", sorted(p.name for p in target.iterdir()))

# --- DIODE validation split ---
diode_root = data_root / "diode"
diode_root.mkdir(parents=True, exist_ok=True)
diode_archive = Path("diode_val.tar.gz")
!wget -q --show-progress -O diode_val.tar.gz "https://diode-dataset.s3.amazonaws.com/val.tar.gz"

assert diode_archive.is_file() and diode_archive.stat().st_size > 0, (
    "DIODE download failed or produced an empty archive"
)
with tarfile.open(diode_archive, "r:gz") as archive:
    archive.extractall(diode_root)
diode_archive.unlink()

assert (diode_root / "val").is_dir(), (
    "data/diode/val was not found after extraction"
)
print("DIODE val ready:", sorted(p.name for p in (diode_root / "val").iterdir()))

## Sign-convention calibration (one real image, one known near/far pair)

In [ ]:
# !python -m src.smoke_test \
#     --checkpoint checkpoints/depth_anything_v2_vits.pth \
#     --model-version v2 \
#     --calibration-image data/da2k/<image> \
#     --near-hw <h> <w> --far-hw <h> <w>

## 20-image smoke test, both tracks

In [ ]:
!python -m src.run_track_a_diode \
    --diode-root data/diode/val \
    --checkpoint-v1 checkpoints/depth_anything_v1_vits.pth \
    --checkpoint-v2 checkpoints/depth_anything_v2_vits.pth \
    --smoke-test

!python -m src.run_track_b_da2k \
    --da2k-root data/da2k \
    --checkpoint-v1 checkpoints/depth_anything_v1_vits.pth \
    --checkpoint-v2 checkpoints/depth_anything_v2_vits.pth \
    --smoke-test

## Full run (drop `--smoke-test`) -- checkpoint predictions periodically if Colab is likely to interrupt a long job

In [ ]:
!python -m src.run_track_a_diode \
    --diode-root data/diode/val \
    --checkpoint-v1 checkpoints/depth_anything_v1_vits.pth \
    --checkpoint-v2 checkpoints/depth_anything_v2_vits.pth

!python -m src.run_track_b_da2k \
    --da2k-root data/da2k \
    --checkpoint-v1 checkpoints/depth_anything_v1_vits.pth \
    --checkpoint-v2 checkpoints/depth_anything_v2_vits.pth

## Apply the pre-registered decision rule

In [ ]:
!python -m src.apply_decision --results-dir results

## Plots

In [ ]:
import json
import matplotlib.pyplot as plt

with open('results/da2k/da2k_comparison.json') as f:
    da2k = json.load(f)
with open('results/diode/diode_comparison.json') as f:
    diode = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].bar(['V1-S', 'V2-S'], [da2k['v1s_accuracy'] * 100, da2k['v2s_accuracy'] * 100])
axes[0].set_title('DA-2K ordinal accuracy (%)')

axes[1].bar(['V1-S', 'V2-S'], [diode['v1s']['abs_rel_mean'], diode['v2s']['abs_rel_mean']])
axes[1].set_title('DIODE AbsRel (lower better)')

axes[2].bar(['V1-S', 'V2-S'], [diode['v1s']['delta1_mean'], diode['v2s']['delta1_mean']])
axes[2].set_title('DIODE delta1 (higher better)')

plt.tight_layout()
plt.savefig('results/comparison_plot.png', dpi=150)
plt.show()